# Cross-Architecture Analysis: why MAAT unlearns Gemma worse than Llama

This notebook turns the §7 claim ("knowledge is *more separable* in Llama")
from an **assertion** into a **measurement**, and adds several mechanistic
diagnostics that increase novelty and reviewer confidence.

Maps directly to the reviews:

| Section | Measures | Addresses |
|---|---|---|
| **A. Gradient separability** | per-module `cos(g_forget, g_retain)` + projection-trigger rate of Eq. 1 | 6Ew8 **Q3**, §7 |
| **B. Subspace overlap** | principal-angle overlap θ between forget/retain gradient subspaces | 6Ew8 **Q3** (exact θ they asked for) |
| **C. Effective rank of forget signal** | participation ratio / stable rank of forget gradients | novelty: tests MAAT's low-rank pruning assumption |
| **D. Per-5W conflict vs answer length** | conflict per category, regressed on answer length | KGpn/LvWk length-vs-causality confound |
| **E. Depth of forgetting** | gold-answer NLL & token-rank (not just greedy FSR) | LvWk/KGpn robustness, boosts confidence |

**Run it once per model** (`MODEL = "llama"`, then `"gemma"`); the final cell
loads both saved JSONs and tabulates the comparison. Requires one GPU that fits
the base model in fp16/bf16 (no 4-bit — quantization noise corrupts the
gradient scoring these diagnostics depend on).


In [1]:
!pip install -q transformers peft datasets accelerate bitsandbytes scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.8 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## Setup — HF login (Kaggle T4x2)


In [7]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

GPUs: 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


## 0. Config


In [18]:
import os, json, math, gc, re
import torch, numpy as np

# ------------------------------------------------------------------ choose one
MODEL = "gemma"          # "llama" | "gemma"
# -----------------------------------------------------------------------------

CFG = {
    "llama": dict(
        base="meta-llama/Llama-3.2-3B",                 # base per released repo
        adapter="Novaspree/factify-3B-adapter",         # finetuned (pre-unlearn) adapter
        dtype=torch.float16,
    ),
    "gemma": dict(
        base="google/gemma-3-4b-it",
        adapter="Novaspree/factify-Gemma3-adapter-1",   # VERIFY this repo id
        dtype=torch.bfloat16,
    ),
}[MODEL]

# MAAT edit band + Phase-1 unlearn target types (must match the method notebooks)
MID_START, MID_END   = 7, 20
UNLEARN_TYPES        = ("down_proj", "up_proj", "q_proj", "v_proj")   # Eq.1 / Phase 1
ALL_LORA_TYPES       = ("down_proj","up_proj","gate_proj","q_proj","k_proj","v_proj","o_proj")

N_SAMPLES   = 80          # forget/retain samples used for gradient diagnostics
SUBSPACE_K  = 24          # samples per module for subspace/effective-rank SVD
SEED        = 42
OUT_DIR     = "results/analysis"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("MODEL:", MODEL, "| base:", CFG["base"])


MODEL: gemma | base: google/gemma-3-4b-it


## 1. Data — forget / retain with 5W labels


In [19]:
from datasets import load_dataset

HF_DATASET = "Novaspree/factify_5K_enriched"

def _load(split_files):
    for f in split_files:
        try:
            return load_dataset(HF_DATASET, data_files=f, split="train")
        except Exception:
            continue
    raise RuntimeError(f"could not load any of {split_files}")

forget_ds = _load(["forget/forget_set_fixed.json", "forget_set_fixed.json"])
retain_ds = _load(["retain/retain_set_fixed.json", "retain_set_fixed.json"])

def take(ds, n):
    idx = np.random.RandomState(SEED).permutation(len(ds))[:n]
    return [ds[int(i)] for i in idx]

forget = take(forget_ds, N_SAMPLES)
retain = take(retain_ds, N_SAMPLES)
print(f"forget={len(forget)} retain={len(retain)} | keys={list(forget[0].keys())}")


forget=80 retain=80 | keys=['question', 'answer', 'pred_answer', 'label', 'rephrases', 'combined_text', 'cluster_id', 'split']


## 2. Model + finetuned adapter (pre-unlearn)


In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(CFG["adapter"] if False else CFG["base"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    CFG["base"], torch_dtype=CFG["dtype"], device_map="auto")
model = PeftModel.from_pretrained(base, CFG["adapter"], is_trainable=True)
model.eval()
DEV = next(model.parameters()).device
print("loaded. device:", DEV)

def build_prompt(q):
    # EXACT prompt used at implantation / by MAAT (base Llama tokenizer has no
    # chat_template, and the Instruct template injects a system-date header the
    # adapter never saw). Hardcode per model; pair with add_special_tokens=False.
    if MODEL == "gemma":
        return ('<bos><start_of_turn>user\n'
                f'{q}<end_of_turn>\n'
                '<start_of_turn>model\n')
    return ('<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
            f'{q}<|eot_id|>'
            '<|start_header_id|>assistant<|end_header_id|>\n\n')

def encode(q, a):
    prompt = build_prompt(q)
    plen = tokenizer(prompt, return_tensors="pt",
                     add_special_tokens=False)["input_ids"].shape[1]
    enc = tokenizer(prompt + a, return_tensors="pt", truncation=True,
                    max_length=512, add_special_tokens=False).to(DEV)
    labels = enc["input_ids"].clone()
    labels[:, :plen] = -100
    return enc, labels


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

loaded. device: cuda:0


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.9.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.10.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.vision_model.encoder.layers.10.self_attn.k_proj.lora_B.default.weight',

## 3. Gradient extraction

For a sample we take the **per-module LoRA-B gradient** in the edit band — the
exact tensors Phase 1 operates on. `requires_grad` is enabled on all targeted
`lora_B` so every module type produces signal (mirrors
`compute_forget_task_vector` in the method notebooks).


In [21]:
import torch.nn as nn

def lora_b_targets(types):
    out = {}
    for name, mod in model.named_modules():
        if not isinstance(mod, nn.Linear):
            continue
        if any(skip in name for skip in ("vision", "multi_modal", "audio")):
            continue
        m = re.search(r"layers\.(\d+)", name)
        if not m or not (MID_START <= int(m.group(1)) <= MID_END):
            continue
        if "lora_B" in name and any(t in name for t in types):
            out[name] = mod
    return out

def module_grads(sample, targets):
    enc, labels = encode(sample["question"], sample["answer"])
    saved = {n: mod.weight.requires_grad for n, mod in targets.items()}
    for mod in targets.values():
        mod.weight.requires_grad_(True)
    model.zero_grad()
    model(**enc, labels=labels).loss.backward()
    grads = {}
    for n, mod in targets.items():
        if mod.weight.grad is not None:
            grads[n] = mod.weight.grad.detach().float().flatten().cpu()
    for n, mod in targets.items():
        mod.weight.requires_grad_(saved[n])
    model.zero_grad()
    return grads

targets_unlearn = lora_b_targets(UNLEARN_TYPES)
targets_all     = lora_b_targets(ALL_LORA_TYPES)
print(f"unlearn targets: {len(targets_unlearn)} | all targets: {len(targets_all)}")
# per-type counts — confirm Llama and Gemma use the same module composition
from collections import Counter
print("by type:", Counter(t for n in targets_all for t in ALL_LORA_TYPES if t in n))

unlearn targets: 48 | all targets: 84
by type: Counter({'q_proj': 12, 'k_proj': 12, 'v_proj': 12, 'o_proj': 12, 'gate_proj': 12, 'up_proj': 12, 'down_proj': 12})


## A. Gradient separability + projection-trigger rate  *(6Ew8 Q3, §7)*

Eq. 1 projects the forget gradient only when it **conflicts** with the retain
gradient (`dot > 0`). Two quantities per architecture:

- **mean `cos(g_f, g_r)`** across the Phase-1 target modules — how aligned the
  forget and retain updates are. Higher ⇒ harder to forget without hurting retain.
- **projection-trigger rate** — fraction of (module, sample-pair) where Eq. 1
  actually projects. A higher rate on Gemma is a *direct mechanistic reason* it
  unlearns less: more of the ascent gets orthogonalised away.


In [22]:
from tqdm.auto import tqdm

def cos(a, b):
    d = (a * b).sum()
    return (d / (a.norm() * b.norm()).clamp_min(1e-12)).item(), d.item()

cos_vals, trigger = [], []
per_type = {t: {"cos": [], "trig": []} for t in UNLEARN_TYPES}

for fs, rs in tqdm(list(zip(forget, retain)), desc="A: separability"):
    gf = module_grads(fs, targets_unlearn)
    gr = module_grads(rs, targets_unlearn)
    for name in gf.keys() & gr.keys():
        c, dot = cos(gf[name], gr[name])
        cos_vals.append(c); trigger.append(1.0 if dot > 0 else 0.0)
        for t in UNLEARN_TYPES:
            if t in name:
                per_type[t]["cos"].append(c); per_type[t]["trig"].append(1.0 if dot > 0 else 0.0)

sec_A = {
    "mean_cos_forget_retain": float(np.mean(cos_vals)),
    "std_cos": float(np.std(cos_vals)),
    "projection_trigger_rate": float(np.mean(trigger)),
    "per_module_type": {t: {"mean_cos": float(np.mean(v["cos"])),
                            "trigger_rate": float(np.mean(v["trig"]))}
                        for t, v in per_type.items() if v["cos"]},
}
print(json.dumps(sec_A, indent=2))


A: separability:   0%|          | 0/80 [00:00<?, ?it/s]

{
  "mean_cos_forget_retain": -0.00046868284192841023,
  "std_cos": 0.04068892734196228,
  "projection_trigger_rate": 0.48098958333333336,
  "per_module_type": {
    "down_proj": {
      "mean_cos": 0.0013886355026355565,
      "trigger_rate": 0.4864583333333333
    },
    "up_proj": {
      "mean_cos": -4.2644649514992734e-05,
      "trigger_rate": 0.47604166666666664
    },
    "q_proj": {
      "mean_cos": -0.003601561837236507,
      "trigger_rate": 0.475
    },
    "v_proj": {
      "mean_cos": 0.00038083961640230274,
      "trigger_rate": 0.4864583333333333
    }
  }
}


## B. Forget/retain subspace overlap θ  *(the exact quantity 6Ew8 requested)*

Per-sample cosine can average out. We build, per module, a **forget-gradient
subspace** (top singular directions of the stacked forget gradients) and a
retain subspace, then measure their overlap via the principal angles:

`overlap = mean(cos^2(principal angles)) ∈ [0, 1]`  — 1 = identical subspaces.

Averaged over modules, this is a single, comparable **separability** number per
architecture — directly what §7 needs.


In [23]:
def stack_grads(samples, targets, k):
    mats = {n: [] for n in targets}
    for s in samples[:k]:
        g = module_grads(s, targets)
        for n, v in g.items():
            mats[n].append(v)
    return {n: torch.stack(vs) for n, vs in mats.items() if vs}  # [k, d]

def top_subspace(mat, r):
    # rows = samples; return orthonormal basis of the top-r right singular vecs
    U, S, Vh = torch.linalg.svd(mat.double(), full_matrices=False)
    r = min(r, Vh.shape[0])
    return Vh[:r].T                                   # [d, r] orthonormal columns

Fm = stack_grads(forget, targets_all, SUBSPACE_K)
Rm = stack_grads(retain, targets_all, SUBSPACE_K)
R_SUB = 8
overlaps = []
for n in Fm.keys() & Rm.keys():
    Uf = top_subspace(Fm[n], R_SUB)
    Ur = top_subspace(Rm[n], R_SUB)
    M = Uf.T @ Ur                                     # [r, r]
    principal_cos2 = torch.linalg.svdvals(M).clamp(0, 1) ** 2
    overlaps.append(principal_cos2.mean().item())

sec_B = {"subspace_overlap_theta": float(np.mean(overlaps)),
         "std": float(np.std(overlaps)), "r": R_SUB, "n_modules": len(overlaps)}
print(json.dumps(sec_B, indent=2))
print("Interpretation: higher θ ⇒ forget & retain live in more overlapping "
      "subspaces ⇒ less separable ⇒ harder clean unlearning.")


{
  "subspace_overlap_theta": 0.013673860069402152,
  "std": 0.021439337923069606,
  "r": 8,
  "n_modules": 84
}
Interpretation: higher θ ⇒ forget & retain live in more overlapping subspaces ⇒ less separable ⇒ harder clean unlearning.


## C. Effective rank of the forget signal  *(novelty: tests the pruning assumption)*

MAAT prunes a **fixed fraction** of MLP dims (0.15) and masks the top-50% of
task-vector rank dims — this only works if the forget signal is **low-rank**.
We measure the *participation ratio* of the forget-gradient spectrum per module:

`PR = (Σ σ_i²)² / Σ σ_i⁴`  — the effective number of active directions.

If Gemma's forget signal has **higher effective rank**, fixed-ratio pruning
removes proportionally less of it — a concrete, novel reason Gemma under-forgets.


In [24]:
def participation_ratio(mat):
    s = torch.linalg.svdvals(mat.double())
    s2 = s ** 2
    return (s2.sum() ** 2 / (s2 ** 2).sum().clamp_min(1e-30)).item()

def stable_rank(mat):
    s = torch.linalg.svdvals(mat.double())
    return ((s ** 2).sum() / (s.max() ** 2).clamp_min(1e-30)).item()

pr = [participation_ratio(Fm[n]) for n in Fm]
sr = [stable_rank(Fm[n]) for n in Fm]
sec_C = {"forget_participation_ratio_mean": float(np.mean(pr)),
         "forget_stable_rank_mean": float(np.mean(sr)),
         "n_samples_per_module": SUBSPACE_K, "n_modules": len(pr)}
print(json.dumps(sec_C, indent=2))


{
  "forget_participation_ratio_mean": 3.1665106070538775,
  "forget_stable_rank_mean": 1.874287771375982,
  "n_samples_per_module": 24,
  "n_modules": 84
}


## D. Per-5W conflict vs answer length  *(KGpn/LvWk: causality or just length?)*

Compute a per-item conflict score against the **mean retain gradient**, group by
5W category, and report it alongside mean answer length. Then regress conflict on
answer length across items. If Why stays high-conflict **after** controlling for
length, difficulty is not purely a length artifact — a targeted rebuttal to the
length-confound critique.


In [25]:
from scipy import stats

# mean retain gradient direction per module (a fixed reference)
gr_mean = {n: torch.stack([Rm[n][i] for i in range(Rm[n].shape[0])]).mean(0)
           for n in Rm}

rows = []
for fs in tqdm(forget, desc="D: per-category"):
    g = module_grads(fs, targets_all)
    dots = [ (g[n] * gr_mean[n]).sum().item() for n in g.keys() & gr_mean.keys() ]
    conflict = float(np.mean([1.0 if d > 0 else 0.0 for d in dots]))   # trigger frac
    ntok = len(tokenizer(fs["answer"], add_special_tokens=False)["input_ids"])
    rows.append({"label": fs.get("label", "?"), "conflict": conflict, "ans_tokens": ntok})

import collections
by_cat = collections.defaultdict(list); tok_by_cat = collections.defaultdict(list)
for r in rows:
    by_cat[r["label"]].append(r["conflict"]); tok_by_cat[r["label"]].append(r["ans_tokens"])

conf = np.array([r["conflict"] for r in rows]); toks = np.array([r["ans_tokens"] for r in rows])
r_len, p_len = stats.pearsonr(toks, conf) if len(rows) > 3 else (float("nan"), float("nan"))

sec_D = {
    "per_category": {c: {"mean_conflict": float(np.mean(v)),
                         "mean_ans_tokens": float(np.mean(tok_by_cat[c]))}
                     for c, v in sorted(by_cat.items())},
    "pearson_conflict_vs_length": {"r": float(r_len), "p": float(p_len)},
}
print(json.dumps(sec_D, indent=2))
print("If Why has high mean_conflict but r(length,conflict) is weak, "
      "the difficulty is not explained by answer length alone.")


D: per-category:   0%|          | 0/80 [00:00<?, ?it/s]

{
  "per_category": {
    "what": {
      "mean_conflict": 0.5124223602484472,
      "mean_ans_tokens": 3.4782608695652173
    },
    "when": {
      "mean_conflict": 0.48917748917748916,
      "mean_ans_tokens": 8.090909090909092
    },
    "where": {
      "mean_conflict": 0.48809523809523814,
      "mean_ans_tokens": 3.6666666666666665
    },
    "who": {
      "mean_conflict": 0.5046296296296295,
      "mean_ans_tokens": 3.5
    },
    "why": {
      "mean_conflict": 0.481829573934837,
      "mean_ans_tokens": 36.526315789473685
    }
  },
  "pearson_conflict_vs_length": {
    "r": -0.04675383260188658,
    "p": 0.6804693046168471
  }
}
If Why has high mean_conflict but r(length,conflict) is weak, the difficulty is not explained by answer length alone.


## E. Depth of forgetting: gold-answer NLL & token-rank  *(beyond greedy FSR)*

FSR only checks whether the *greedy* decode contains the answer. A model can hide
the answer from greedy decoding while keeping it high-probability. We report the
**NLL** and **mean token-rank** of the gold answer.

Run this cell with the **finetuned** adapter (loaded above) to get the pre-unlearn
baseline, then set `ADAPTER_OVERRIDE` to your **unlearned** adapter and re-run:
a genuine forget should *raise NLL and rank a lot*; shallow suppression barely moves them.


In [26]:
ADAPTER_OVERRIDE = None      # e.g. "/kaggle/working/lora_adapter_unlearned" for post-unlearn

@torch.no_grad()
def nll_and_rank(sample):
    enc, labels = encode(sample["question"], sample["answer"])
    out = model(**enc)
    logits = out.logits.float()[0][:-1]         # predict token t+1
    tgt = enc["input_ids"][0][1:]
    lab = labels[0][1:]
    mask = lab != -100
    if mask.sum() == 0:
        return None
    lp = torch.log_softmax(logits[mask], dim=-1)
    gold = tgt[mask]
    nll = -lp[range(len(gold)), gold].mean().item()
    ranks = (logits[mask] > logits[mask][range(len(gold)), gold].unsqueeze(1)).sum(1)
    return nll, ranks.float().mean().item()

vals = [nll_and_rank(s) for s in tqdm(forget, desc="E: forget NLL")]
vals = [v for v in vals if v]
sec_E = {"adapter": ADAPTER_OVERRIDE or CFG["adapter"],
         "forget_gold_nll_mean": float(np.mean([v[0] for v in vals])),
         "forget_gold_rank_mean": float(np.mean([v[1] for v in vals]))}
print(json.dumps(sec_E, indent=2))


E: forget NLL:   0%|          | 0/80 [00:00<?, ?it/s]

{
  "adapter": "Novaspree/factify-Gemma3-adapter-1",
  "forget_gold_nll_mean": 0.5692145182888048,
  "forget_gold_rank_mean": 3.533680255827494
}


## 4. Save this model's report


In [27]:
report = {"model": MODEL, "base": CFG["base"], "n_samples": N_SAMPLES,
          "A_separability": sec_A, "B_subspace_overlap": sec_B,
          "C_effective_rank": sec_C, "D_category_length": sec_D,
          "E_depth_of_forgetting": sec_E}
path = f"{OUT_DIR}/analysis_{MODEL}.json"
json.dump(report, open(path, "w"), indent=2)
print("saved ->", path)


saved -> results/analysis/analysis_gemma.json


## 5. Cross-architecture comparison  *(run after both `llama` and `gemma` are saved)*

Loads both reports and prints the headline table for §7 / the rebuttal.


In [28]:
import json, os
paths = {m: f"{OUT_DIR}/analysis_{m}.json" for m in ("llama", "gemma")}
if all(os.path.exists(p) for p in paths.values()):
    R = {m: json.load(open(p)) for m, p in paths.items()}
    def row(label, fn):
        print(f"{label:<38}{fn(R['llama']):>14}{fn(R['gemma']):>14}")
    print(f"{'metric':<38}{'Llama':>14}{'Gemma':>14}")
    print("-" * 66)
    row("mean cos(g_f, g_r)  (A)",       lambda r: f"{r['A_separability']['mean_cos_forget_retain']:.4f}")
    row("projection-trigger rate  (A)",  lambda r: f"{r['A_separability']['projection_trigger_rate']:.3f}")
    row("subspace overlap θ  (B)",       lambda r: f"{r['B_subspace_overlap']['subspace_overlap_theta']:.4f}")
    row("forget participation ratio (C)",lambda r: f"{r['C_effective_rank']['forget_participation_ratio_mean']:.2f}")
    row("forget gold NLL  (E)",          lambda r: f"{r['E_depth_of_forgetting']['forget_gold_nll_mean']:.3f}")
    print("-" * 66)
    print("Expected story if §7 holds: Gemma shows higher cos, higher trigger "
          "rate, higher θ, higher participation ratio ⇒ less separable, "
          "higher-rank forget signal ⇒ harder to unlearn cleanly.")
else:
    print("Run the notebook with MODEL='llama' and MODEL='gemma' first.")


metric                                         Llama         Gemma
------------------------------------------------------------------
mean cos(g_f, g_r)  (A)                       0.0046       -0.0005
projection-trigger rate  (A)                   0.559         0.481
subspace overlap θ  (B)                       0.0157        0.0137
forget participation ratio (C)                  1.35          3.17
forget gold NLL  (E)                           0.147         0.569
------------------------------------------------------------------
Expected story if §7 holds: Gemma shows higher cos, higher trigger rate, higher θ, higher participation ratio ⇒ less separable, higher-rank forget signal ⇒ harder to unlearn cleanly.


## How to use these results in the rebuttal

- **6Ew8 Q3 / §7** → Sections **A** and **B** give the measured separability
  (θ, cos, projection-trigger rate) they asked for. Replace the asserted "more
  separable in Llama" with these numbers, or, if they don't separate the models,
  downgrade §7 to a hypothesis (as the reviewer requested).
- **Novelty (KGpn)** → Section **C** (effective rank of the forget signal) and
  the projection-trigger analysis are new mechanistic findings that explain *why*
  the combination of techniques behaves differently across architectures — not
  just that it does.
- **Length confound (KGpn/LvWk)** → Section **D** separates category effects from
  answer length.
- **Robust forgetting (all three)** → Section **E** shows depth of forgetting
  beyond greedy FSR; pair with the paraphrase/relearning harness.
